# eeg2img on Colab — data preparation

This notebook only *launches* the repository's scripts; it contains no logic of its own.
It downloads Alljoined-1.6M, preprocesses it into a compact store, computes the frozen
DINOv2-giant image targets, and saves the result to Google Drive for later sessions.

**Before running**

1. Runtime → Change runtime type → **T4 GPU**.
2. The repository is private. Create a GitHub fine-grained token with read-only *Contents*
   access to it, then add it under the key icon (Secrets) as `GH_TOKEN` and enable notebook access.
3. Allow about 1 hour and 20 GB of Colab disk. About 3.8 GB is written to your Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO = "moh246/eeg2image-neurips26"
DRIVE_DIR = "/content/drive/MyDrive/eeg2img"          # persistent
WORK_DIR = "/content/work"                             # fast local disk, lost at session end
STORE = f"{WORK_DIR}/stores/alljoined16m_v1"

import os, subprocess
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(WORK_DIR, exist_ok=True)
if not os.path.isdir("/content/eeg2image-neurips26"):
    token = userdata.get("GH_TOKEN")
    subprocess.run(["git", "clone", "--quiet", f"https://{token}@github.com/{REPO}.git",
                    "/content/eeg2image-neurips26"], check=True)
%cd /content/eeg2image-neurips26
!git pull --quiet && git log -1 --format='%h %s'

In [ ]:
%pip install --quiet -e ".[data,targets]"
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Download and preprocess the EEG

Alljoined-1.6M is licensed **CC-BY-NC-ND-4.0** (non-commercial). Setting
`ALLJOINED_ACCEPT_LICENCE=1` below is your acceptance of that licence.
Both steps resume if the session is interrupted.

In [ ]:
%env ALLJOINED_ACCEPT_LICENCE=1
!python scripts/prepare_alljoined.py --config configs/data/alljoined16m.yaml \
    data.root={WORK_DIR}/data data.store={STORE}

## 2. Compute the DINOv2-giant targets

Uses the organisers' own extractor so the targets match the competition gallery.

In [ ]:
!python scripts/embed_images.py --config configs/data/alljoined16m.yaml --device cuda \
    data.root={WORK_DIR}/data data.store={STORE} data.embeddings={STORE}/embeddings.npy

## 3. Save to Drive and record checksums

Only the final store is copied (not the raw download or the intermediate shards).
Paste the printed summary back so it can be recorded in `data/README.md`.

In [ ]:
import hashlib, json, shutil
from pathlib import Path

src, dst = Path(STORE), Path(DRIVE_DIR) / "stores" / "alljoined16m_v1"
dst.mkdir(parents=True, exist_ok=True)
summary = {}
for name in ["eeg.npy", "index.npz", "images.json", "meta.json", "embeddings.npy"]:
    digest = hashlib.sha256()
    with open(src / name, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    shutil.copy2(src / name, dst / name)
    summary[name] = {"bytes": (src / name).stat().st_size, "sha256": digest.hexdigest()[:16]}
meta = json.loads((src / "meta.json").read_text())
summary["counts"] = {k: meta[k] for k in ("n_recordings", "n_subjects", "n_events", "n_images")}
summary["versions"] = meta["provenance"]
print(json.dumps(summary, indent=2))